In [1]:
"""
Only FWI - Jupyter single-GPU version
"""

import os
import yaml
import shutil
from pathlib import Path

import numpy as np
import torch
import tqdm

from deepwave import scalar

#from ifwitorch.loss import *
#from ifwitorch.signal import filter_dw, filter
#from diffifwi.diffusion import *
#from diffifwi.plots import *
#from diffifwi.utils import *
#from utils import *
#from general import *
from utils.utils import *

/home/sadalyom/.conda/envs/DAS_Denoising_Torch/lib/python3.8/site-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/sadalyom/.conda/envs/DAS_Denoising_Torch/lib/python3.8/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/home/sadalyom/.conda/envs/DAS_Denoising_Torch/lib/python3.8/site-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/sadalyom/.conda/envs/DAS_Denoising_Torch/lib/python3.8/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


## notebook arguments

In [2]:
class Args:
    def __init__(self):
        self.configure = "utils/viking_wlr.yml"
        self.savepath = "DIIP_Viking"
        self.loss = "cs"
        self.std = ["vp", "400"]
        self.mean = ["vp", "0"]
        self.layers = 2
        self.features = 128
        self.use_hash = True
        self.use_multiscale = True
        self.model_split = True

args = Args()

## setup and config loading

In [3]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
master = True
size = 1
rank = 0

# Load configure file
config_file = args.configure
if not os.path.exists(config_file):
    raise FileNotFoundError(f"File {config_file} not found")

with open(config_file) as file:
    cfg = yaml.load(file, Loader=yaml.FullLoader)
    cfg_obs = cfg["observed"]
    cfg = cfg["inversion"]
    cfg_wavelet = cfg["wavelet"]
    cfg_network = cfg["networks"]

# Seed
torch.manual_seed(cfg["training"].get("seed", 42))

# Paths
#save_root = args.savepath
#path = initialize_paths(save_root)

# Optional: save a copy of the notebook-converted code manually if needed
print(f"Using device: {device}")
#print(f"Save root: {save_root}")

Using device: cuda


## Load models and acquisition

In [4]:
models_true = {k: np.load(v) for k, v in cfg_obs["geom"]["models"].items()}
models_init = {k: np.load(v) for k, v in cfg["geom"]["models"].items()}

est_shape = models_true[list(models_true.keys())[0]].shape

end_nt = int(cfg_wavelet["endt"] / cfg_wavelet["dt"])
wave = initialize_wavelet(cfg_wavelet)[:end_nt]
sources, receivers = load_acquisition_data(cfg, est_shape[1])

print(f"Sources: {sources.shape}")
print(f"Receivers: {receivers.shape}")

Load wavelet from data/Viking_data/new_wavelet.npy
Loading sources from data/Viking_data/viking_source_deepwave_wlr.npy
Loading receivers from data/Viking_data/viking_receiver_deepwave_wlr.npy
Sources: (1001, 1, 2)
Receivers: (1001, 120, 2)


## criterion and optimizer

In [5]:
criterion = {
    "l2": l2,
    "cs": cs,
    "rtm": rtm,
}[args.loss]

init_vp = models_init["vp"]
dh = cfg["geom"]["dh"]
dt = cfg["wavelet"]["dt"]
spatial_order = cfg["solver"]["equation"]["spatial_order"]
pml_width = [20, 20, 20, 20] if not cfg["solver"]["free_surface"] else [20, 20, 0, 20]

epoch0 = 0
total_epochs = cfg["training"]["epochs"]

init_vp = torch.from_numpy(init_vp).to(device)
v = torch.nn.Parameter(init_vp.clone())

opt = torch.optim.Adam([v], lr=15, eps=float(cfg["training"]["eps"]))

extent = [0, est_shape[1] * dh, dh * est_shape[0], 0]
kwargs_show = dict(
    vmin=models_true["vp"].min(),
    vmax=models_true["vp"].max(),
    cmap="seismic",
    aspect="auto",
    extent=extent,
)

std = parse_std_args(args.std)["vp"]
mean = parse_std_args(args.mean)["vp"]

cfg["args"] = vars(args)
#with open(os.path.join(save_root, "configures.yml"), "w") as file:
#    yaml.dump(cfg, file)

## multiscale and dataset

In [6]:
frequency_list = cfg["multiscale"]["frequencies"]
epochs = cfg["training"]["epochs"]
epoch_per_freq = epochs // len(frequency_list) if args.use_multiscale else epochs

rmse = lambda inv, true: np.sqrt(np.mean((inv - true) ** 2))
LOSS = []
RMSE = []
SHOT_NO = []

grad_combine = torch.zeros(init_vp.shape, dtype=torch.float32, device=device) if args.model_split else None

msplit, left, right, sources_split, receivers_split = split_model(
    [init_vp.cpu().numpy()],
    sources,
    receivers,
    one_side_expand=50
)

print("Original model shape:", init_vp.shape, "Split model shape", msplit.shape[-2:])

dataset = OBSDataset(
    cfg_obs["observed_data_path"],
    sources,
    receivers,
    Preload=True,
    endnt=end_nt
)

# Single-GPU notebook: batch size is not divided by number of GPUs
# If multiscale uses different batch sizes per frequency, it will be rebuilt inside loop.

Max offset in x: 259.0
Original model shape: torch.Size([320, 2281]) Split model shape (320, 359)


In [7]:
from torchsummary import summary
from Model.Module22_Diff_31 import *
import  torch
from utils.Diff_Utils import *


#Learning Schedule
def lr_lambda(epoch):
    return 10
def lr_lambda1(epoch):
    return 1/2

sim_lr = 1e-4
# define model and diffusion
device = "cuda" if torch.cuda.is_available() else "cpu"
netD = UNetModel(
    in_channels=1,
    model_channels=64,
    out_channels=1,
    num_res_blocks=4,
    attention_resolutions=(8, 16),
    dropout=0,
    channel_mult=(1, 2, 2),
    conv_resample=True,
    num_heads=4
)

### init the hyper-parameters of netD ###
#netD = netD.apply(lambda m: weights_init(m, leak_value))
netD = netD.to(device)


# Optimizer for Diffusion network.
#optim_sPre = optim.Adam(netD.parameters(),lr=sim_lr,betas=(0.5, 0.99), \
#                    eps=1e-8, weight_decay=0)


timesteps = 1000
gaussian_diffusion = GaussianDiffusion(timesteps=timesteps)



"""Compute number of parameters"""
s  = sum(np.prod(list(p.size())) for p in netD.parameters())
print ('Number of netD params: %d' % s)

#summary(netD,[(1,nt,num_receivers_per_shot),(1,nt,num_receivers_per_shot)])


############## Init  ################# 
SNR = 0.0
SSIM = 0.0
Loss = 0.0
ERROR = 0.0

pa = 'Model/netD_ema256_MorePatches_FaultLinear_all'
netD.load_state_dict(torch.load(pa, weights_only=True))


water_bottom = 0
print("Water bottom =", water_bottom)

Number of netD params: 9941761


/home/sadalyom/.conda/envs/DAS_Denoising_Torch/lib/python3.8/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Water bottom = 0


In [8]:
from contextlib import nullcontext
from tqdm import trange, tqdm
import os
import time
import numpy as np
import torch

# ============================================================
# DIIP-FWI: DIIP OUTSIDE shot loop
# ============================================================

# ----------------------------
# DIIP settings
# ----------------------------
t_start, t_end = 100, 2
warm_epochs = 20
power_t = 1.0

Kf = 10
eta_ddim = 0.0

patch_chunk = 1
use_checkpoint = True
use_amp = False
amp_dtype = torch.float16

lr_start, lr_end = 1e-3, 1e-3
power_lr = 2.0
grad_clip = 1.0

min_vel = 1200.0
max_vel = 5000.0
mgsi = 4

w1, w2 = 256, 256
o1, o2 = w1 // 2, w2 // 2

# ------------------------------------------------------------
# Folder name
# ------------------------------------------------------------
folder_name = (
    f"results/DIIP_FWI_Viking_Kf{Kf}_t{t_start}-{t_end}_"
    f"w{warm_epochs}_lr{lr_start:.0e}-{lr_end:.0e}_FixedSchedule"
)

print("Saving to:", folder_name)
path = initialize_paths(folder_name)

# ----------------------------
# Freeze diffusion model
# ----------------------------
netD.eval()
for p in netD.parameters():
    p.requires_grad_(False)

# ----------------------------
# Helpers
# ----------------------------
def vel_to_patches(v, min_vel, max_vel):
    v_n = normalize_vel(v, min_vel, max_vel)
    return patch_torch(
        v_n, w1, w2, o1, o2
    ).reshape(-1, 1, w1, w2)


def patches_to_vel(x0_hat, H, W, min_vel, max_vel):
    v_new_n = patch_inv_torch_gaussian(
        x0_hat.reshape(-1, w1 * w2),
        H, W,
        w1, w2,
        o1, o2,
        sigma1=w1 / 6,
        sigma2=w2 / 6,
    )

    return denormalize_vel(
        v_new_n,
        min_vel,
        max_vel
    )

# ----------------------------
# Optimize delta, not velocity
# ----------------------------
v_base = torch.from_numpy(
    models_init["vp"]
).float().to(device)

H, W = v_base.shape

v0_p = vel_to_patches(
    v_base,
    min_vel,
    max_vel
)

with torch.no_grad():
    x_t_inv, ts = gaussian_diffusion.ddim_invert_x0_to_xt(
        model=netD,
        x0=v0_p,
        t_start=t_start,
        K=min(Kf, max(2, t_start)),
        clip_denoised=False,
        use_amp=False,
    )

delta = torch.zeros_like(
    x_t_inv,
    device=device,
    requires_grad=True
)

opt = torch.optim.Adam(
    [delta],
    lr=lr_start,
    betas=(0.5, 0.9),
    eps=1e-12
)

# ============================================================
# Main loop
# ============================================================
pbar = trange(epoch0, epochs)

obsloader = None
prev_freq_idx = None

global_start = time.time()
epoch_times = []

for epoch in pbar:

    epoch_start = time.time()

    opt.zero_grad(set_to_none=True)

    if args.model_split:
        grad_combine.zero_()

    # ========================================================
    # Multiscale frequency selection
    # ========================================================
    freq_idx_raw, freq_epoch_raw = divmod(
        epoch,
        epoch_per_freq
    )

    if args.use_multiscale:
        n_freq = len(frequency_list)

        freq_idx = min(
            freq_idx_raw,
            n_freq - 1
        )

        freq_this_epoch = frequency_list[freq_idx]
        beyond_last_scale = freq_idx_raw >= n_freq

    else:
        n_freq = 1
        freq_idx = 0
        freq_this_epoch = "all"
        beyond_last_scale = False

    # ========================================================
    # Timestep and LR schedule
    # ========================================================
    warm_epochs_scale = min(
        warm_epochs,
        epoch_per_freq
    )

    if beyond_last_scale:

        freq_epoch = epoch_per_freq - 1
        t_int = t_end
        lr_now = float(lr_end)

    else:

        freq_epoch = freq_epoch_raw

        if freq_epoch < warm_epochs_scale:

            p = freq_epoch / max(
                1,
                warm_epochs_scale - 1
            )

            t_int = int(round(
                t_start +
                (t_end - t_start) *
                (p ** power_t)
            ))

            lr_now = float(
                lr_start +
                (lr_end - lr_start) *
                (p ** power_lr)
            )

        else:

            t_int = t_end
            lr_now = float(lr_end)

    K_now = min(
        Kf,
        max(2, t_int)
    )

    for pg in opt.param_groups:
        pg["lr"] = lr_now

    # ========================================================
    # Rebuild loader only at true frequency-band transitions
    # ========================================================
    reset_loader = (
        obsloader is None
        or (
            freq_idx != prev_freq_idx
            and not beyond_last_scale
        )
    )

    if reset_loader:

        shots_per_batch = (
            int(cfg["training"]["batchsize"][freq_idx])
            if args.use_multiscale
            else int(cfg["training"]["batchsize"])
        )

        obsloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=shots_per_batch,
            shuffle=True
        )

        bps = 1
        steps_per_epoch = shots_per_batch // bps

        tqdm.write(f"Inversion single GPU, batchsize {shots_per_batch}")
        tqdm.write(f"Steps per epoch: {steps_per_epoch}, shots per step: {bps}")
        tqdm.write(
            f"Reset schedule for scale {freq_idx + 1}/{n_freq} | "
            f"F = {freq_this_epoch}"
        )

        prev_freq_idx = freq_idx

    # ========================================================
    # Refresh inversion anchor once per epoch
    # ========================================================
    with torch.no_grad():

        x_t_inv, ts = gaussian_diffusion.ddim_invert_x0_to_xt(
            model=netD,
            x0=v0_p,
            t_start=t_int,
            K=K_now,
            clip_denoised=False,
            use_amp=False,
        )

    # ========================================================
    # DIIP outside shot loop: decode once per epoch
    # ========================================================
    ctx = (
        torch.cuda.amp.autocast(dtype=amp_dtype)
        if use_amp
        else nullcontext()
    )

    with ctx:

        x_t = x_t_inv + delta

        x0_hat = gaussian_diffusion.ddim_x0_from_xt_with_ts_chunked(
            model=netD,
            x_t=x_t,
            ts=ts,
            eta=eta_ddim,
            clip_denoised=False,
            patch_chunk=patch_chunk,
            use_checkpoint=use_checkpoint,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
        )

        v_full = patches_to_vel(
            x0_hat,
            H,
            W,
            min_vel,
            max_vel
        )

        v_full = torch.clamp(
            v_full,
            min=min_vel,
            max=max_vel
        )

        if water_bottom > 0:
            v_full = v_full.clone()
            v_full[:water_bottom, :] = 1500.0

    # ========================================================
    # Get one batch
    # ========================================================
    obs_this, _, _, shot_nos = next(iter(obsloader))

    pbar.set_description(
        f"Epoch {epoch + 1}/{epochs} | "
        f"F{freq_this_epoch} Hz | "
        f"scale {freq_idx + 1}/{n_freq} | "
        f"freq_epoch {freq_epoch + 1}/{epoch_per_freq} | "
        f"t{t_int} | "
        f"K{K_now}"
    )

    last_loss = None
    last_obs = None
    last_syn = None

    grad_v_full = torch.zeros_like(v_full)

    # ========================================================
    # Shot loop: accumulate velocity gradient
    # ========================================================
    for step in range(steps_per_epoch):

        drange = slice(
            step * bps,
            (step + 1) * bps
        )

        shot_nos_step = shot_nos[drange].numpy()

        if args.model_split:

            mrange = slice(
                left[shot_nos_step][0],
                right[shot_nos_step][0]
            )

            vp_this = v_full[:, mrange]

            src = torch.from_numpy(
                sources_split[shot_nos_step]
            ).to(device)

            rec = torch.from_numpy(
                receivers_split[shot_nos_step]
            ).to(device)

        else:

            vp_this = v_full

            src = torch.from_numpy(
                sources[shot_nos_step]
            ).to(device)

            rec = torch.from_numpy(
                receivers[shot_nos_step]
            ).to(device)

        model_dtype = vp_this.dtype

        if wave.ndim == 3:

            wavelet = torch.from_numpy(
                wave[shot_nos_step]
            ).to(
                device=device,
                dtype=model_dtype
            )

        else:

            wavelet = (
                torch.from_numpy(wave)
                .reshape(1, 1, -1)
                .repeat(src.shape[0], 1, 1)
                .to(
                    device=device,
                    dtype=model_dtype
                )
            )

        obs = obs_this[drange].to(
            device=device,
            dtype=model_dtype
        )

        syn = scalar(
            vp_this.T,
            dh,
            dt,
            source_amplitudes=wavelet,
            source_locations=src,
            receiver_locations=rec,
            pml_width=pml_width,
            pml_freq=8.0,
            accuracy=spatial_order,
            model_gradient_sampling_interval=mgsi,
        )[-1]

        if args.use_multiscale:

            syn = filter_dw(
                syn,
                dt=dt,
                freqs=freq_this_epoch,
                forder=4,
                axis=-1
            )

            obs = filter_dw(
                obs,
                dt=dt,
                freqs=freq_this_epoch,
                forder=4,
                axis=-1
            )

        loss = criterion(
            syn,
            obs
        )

        grad_this = torch.autograd.grad(
            loss,
            v_full,
            retain_graph=True,
            create_graph=False,
            allow_unused=False,
        )[0]

        grad_v_full += grad_this

        last_loss = loss
        last_obs = obs
        last_syn = syn

    # ========================================================
    # Backprop through DIIP once
    # ========================================================
    v_full.backward(
        gradient=grad_v_full
    )

    torch.nn.utils.clip_grad_norm_(
        [delta],
        grad_clip
    )

    opt.step()

    torch.cuda.empty_cache()

    inv_np = v_full.detach().cpu().numpy()

    RMSE.append(
        rmse(inv_np, models_true["vp"])
    )

    LOSS.append(
        last_loss.detach().item()
    )

    # ========================================================
    # Save
    # ========================================================
    if (
        epoch % cfg["training"]["show_every"] == 0
        or epoch == epochs - 1
    ):

        show_model(
            inv_np,
            save_path=os.path.join(
                path["figures"],
                f"inv_vp_epoch{epoch}_t{t_int}"
            ),
            **kwargs_show
        )

        np.save(
            os.path.join(
                path["model"],
                f"inv_vp_epoch{epoch}_t{t_int}.npy"
            ),
            inv_np
        )

        np.save(
            os.path.join(
                path["model"],
                "delta_current.npy"
            ),
            delta.detach().cpu().numpy()
        )

        np.save(
            os.path.join(
                folder_name,
                "loss.npy"
            ),
            np.array(LOSS)
        )

        np.save(
            os.path.join(
                folder_name,
                "rmse.npy"
            ),
            np.array(RMSE)
        )

        np.save(
            os.path.join(
                path["data"],
                f"obs_epoch{epoch}_t{t_int}_r{rank}.npy"
            ),
            last_obs.detach().cpu().numpy()
        )

        np.save(
            os.path.join(
                path["data"],
                f"syn_epoch{epoch}_t{t_int}_r{rank}.npy"
            ),
            last_syn.detach().cpu().numpy()
        )

    # ========================================================
    # Timing
    # ========================================================
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)

    elapsed = time.time() - global_start
    avg_epoch_time = np.mean(epoch_times)
    remaining_epochs = epochs - (epoch + 1)
    eta = avg_epoch_time * remaining_epochs

    tqdm.write(
        f"Epoch {epoch} | "
        f"freq_idx_raw {freq_idx_raw} | "
        f"freq_idx {freq_idx} | "
        f"freq_epoch {freq_epoch} | "
        f"F {freq_this_epoch} | "
        f"t_int {t_int} | "
        f"K {K_now} | "
        f"lr {lr_now:.3e} | "
        f"Loss {last_loss.detach().item():.6e} | "
        f"epoch time {epoch_time/60:.2f} min | "
        f"avg {avg_epoch_time/60:.2f} min/epoch | "
        f"elapsed {elapsed/60:.2f} min | "
        f"ETA {eta/60:.2f} min"
    )

# ============================================================
# Final save
# ============================================================
np.save(
    os.path.join(
        folder_name,
        f"shot_nos_r{rank}.npy"
    ),
    np.array(SHOT_NO)
)

total_time = time.time() - global_start

print(
    f"Done. Total inversion time: "
    f"{total_time/60:.2f} min "
    f"({total_time/3600:.2f} hours)"
)

Saving to: results/DIIP_FWI_Viking_Kf10_t100-2_w20_lr1e-03-1e-03_FixedSchedule


/home/sadalyom/.conda/envs/DAS_Denoising_Torch/lib/python3.8/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at /opt/conda/conda-bld/pytorch_1699449185072/work/aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,
  0%|                                                   | 0/401 [00:00<?, ?it/s]

Inversion single GPU, batchsize 96
Steps per epoch: 96, shots per step: 1
Reset schedule for scale 1/8 | F = [3.0, 6.0]


Epoch 1/401 | F[3.0, 6.0] Hz | scale 1/8 | freq_epoch 1/50 | t100 | K10:   0%| |

Epoch 0 | freq_idx_raw 0 | freq_idx 0 | freq_epoch 0 | F [3.0, 6.0] | t_int 100 | K 10 | lr 1.000e-03 | Loss 8.986387e-01 | epoch time 0.80 min | avg 0.80 min/epoch | elapsed 0.80 min | ETA 320.86 min


Epoch 2/401 | F[3.0, 6.0] Hz | scale 1/8 | freq_epoch 2/50 | t95 | K10:   0%| | 

... (795 outputs left out)

Epoch 400/401 | F[3.0, 20.0] Hz | scale 8/8 | freq_epoch 50/50 | t2 | K2: 100%|▉

Epoch 399 | freq_idx_raw 7 | freq_idx 7 | freq_epoch 49 | F [3.0, 20.0] | t_int 2 | K 2 | lr 1.000e-03 | Loss 4.224708e-01 | epoch time 1.28 min | avg 0.85 min/epoch | elapsed 338.89 min | ETA 0.85 min


Epoch 401/401 | F[3.0, 20.0] Hz | scale 8/8 | freq_epoch 50/50 | t2 | K2: 100%|█

Epoch 400 | freq_idx_raw 8 | freq_idx 7 | freq_epoch 49 | F [3.0, 20.0] | t_int 2 | K 2 | lr 1.000e-03 | Loss 3.425145e-01 | epoch time 1.28 min | avg 0.85 min/epoch | elapsed 340.17 min | ETA 0.00 min
Done. Total inversion time: 340.17 min (5.67 hours)


## Plotting the inverted model

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
%matplotlib inline

cm = ListedColormap(np.fromfile("utils/colorbar_ind.r@", dtype=np.float32).reshape(3,256).T)

iteer = 400
v_init_path = "data/Viking_data/Initial_v.npy"
diip_path = f"{folder_name}/model/inv_vp_epoch{iteer}_t2.npy"

v_init = np.load(v_init_path)
diip_model = np.load(diip_path)

# ------------------------------------------------------------
# Grid spacing
# ------------------------------------------------------------
dx = 12.5
dz = 12.5

# ------------------------------------------------------------
# Color limits
# ------------------------------------------------------------
vmin = 1200
vmax = 4800


# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------
nz, nx = v_init.shape
extent = [0, nx * dx, nz * dz, 0]

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 20,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
})

def plot_panel(ax, model, panel_label, title, show_ylabel=False):
    im = ax.imshow(
        model,
        cmap=cm,
        vmin=vmin,
        vmax=vmax,
        aspect="auto",
        extent=extent,
    )

    ax.set_xlabel("Position (m)")

    if show_ylabel:
        ax.set_ylabel("Depth (m)")
        ax.tick_params(axis='y', labelleft=True)   # ensure visible
    else:
        ax.set_ylabel("")
        ax.tick_params(axis='y', left=False, labelleft=False)  # clean removal

    ax.set_title(title, pad=10)

    return im

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1, 2,
    figsize=(21, 6),
    sharey=True,
    constrained_layout=True
)

im0 = plot_panel(axes[0], v_init, "a)", "Initial", show_ylabel=True)
im1 = plot_panel(axes[1], diip_model, "b)", "DIIP")

# Shared colorbar
cbar = fig.colorbar(im1, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Velocity (m/s)")

plt.show()
